---

Import everything again.

---

In [ ]:
import os
import json
import shutil
import hashlib
import logging
import re
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import yaml
from PIL import Image
from PIL import UnidentifiedImageError
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

print("All required libraries imported successfully")

---

Configure all paths again.

---

In [ ]:
# Project root
PROJECT_ROOT = Path.cwd().parents[1]

# Configuration
CONFIG_DIR = PROJECT_ROOT / "config"
YAML_PATH = CONFIG_DIR / "download_sheet.yaml"

# Dataset directories
DATASET_DIR = PROJECT_ROOT / "datasets"

RAW_DIR = DATASET_DIR / "raw"
VALIDATED_DIR = DATASET_DIR / "validated"
INVALID_DIR = DATASET_DIR / "invalid"
PROCESSED_DIR = DATASET_DIR / "processed"
INTERIM_DIR = DATASET_DIR / "interim"
DATASET_V2_DIR = DATASET_DIR / "dataset_v2"
REPORT_DIR = DATASET_DIR / "reports"

# Display paths
print(f"Project Root : {PROJECT_ROOT}")
print(f"Config       : {CONFIG_DIR}")
print(f"Raw          : {RAW_DIR}")
print(f"Validated    : {VALIDATED_DIR}")
print(f"Invalid      : {INVALID_DIR}")
print(f"Processed    : {PROCESSED_DIR}")
print(f"Interim      : {INTERIM_DIR}")
print(f"Dataset V2   : {DATASET_V2_DIR}")
print(f"Reports      : {REPORT_DIR}")
print(f"YAML File    : {YAML_PATH}")

---

Setting up the log to check activities

---

In [ ]:
# --- Logging setup ---
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = REPORT_DIR / f"validation_log_{timestamp}.log"

logger = logging.getLogger("validation")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if cell is re-run

file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(stream_handler)

logger.info("Validation run started.")
logger.info(f"Log file created at: {LOG_PATH}")

# Structured results collector — will become the CSV report later
validation_results = []

---

Read the config file again

---

In [ ]:
with open(YAML_PATH, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

print(f"Successfully loaded {YAML_PATH.name}")
print(f"Datasets found: {len(dataset_config)}")
logger.info(f"Successfully loaded YAML config from {YAML_PATH}")

---

Extract dataset entries

---

In [ ]:
datasets = dataset_config["datasets"]
dataset_names = list(datasets.keys())

logger.info(f"Extracted {len(dataset_names)} dataset entries from YAML.")
print(f"Total datasets in YAML: {len(dataset_names)}")
for name, meta in datasets.items():
    print(f"  - {name} | source: {meta.get('source')} | status: {meta.get('status')}")

---

Check if all data successfully downloaded

---

In [ ]:
folder_check_results = []

for name, meta in datasets.items():
    status = meta.get("status")
    source = meta.get("source")
    folder_name = meta.get("folder")
    expected_path = RAW_DIR / source / folder_name

    if status == "rejected":
        logger.info(f"[SKIPPED] {name} — status is 'rejected' in YAML, not validated.")
        folder_check_results.append({
            "dataset": name,
            "path": str(expected_path),
            "exists": None,
            "file_count": None,
            "verdict": "SKIPPED"
        })
        continue

    if not expected_path.exists():
        logger.warning(f"[MISSING] {name} — expected folder not found: {expected_path}")
        folder_check_results.append({
            "dataset": name,
            "path": str(expected_path),
            "exists": False,
            "file_count": 0,
            "verdict": "MISSING"
        })
        continue

    file_count = sum(1 for f in expected_path.rglob("*") if f.is_file())

    if file_count == 0:
        logger.warning(f"[EMPTY] {name} — folder exists but contains no files: {expected_path}")
        verdict = "EMPTY"
    else:
        logger.info(f"[OK] {name} — found {file_count} files in {expected_path}")
        verdict = "OK"

    folder_check_results.append({
        "dataset": name,
        "path": str(expected_path),
        "exists": True,
        "file_count": file_count,
        "verdict": verdict
    })

folder_check_df = pd.DataFrame(folder_check_results)
print(folder_check_df.to_string(index=False))

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}
TABULAR_EXTS = {".csv", ".xlsx", ".xls"}
TEXT_LABEL_EXTS = {".txt", ".json", ".xml"}
METADATA_EXTS = {".yaml", ".yml", ".md", ".py"}
METADATA_FILENAMES = {"license", "readme", "readme.txt", "changelog", "notice"}


def check_file(filepath: Path):
    """Return (status, reason) for a single file."""
    ext = filepath.suffix.lower()
    stem_lower = filepath.name.lower()

    try:
        # Check filename-based metadata first (LICENSE, README have no extension)
        if stem_lower in METADATA_FILENAMES:
            with open(filepath, "r", encoding="utf-8", errors="strict") as f:
                f.read()
            return "METADATA", None

        if ext in IMAGE_EXTS:
            with Image.open(filepath) as img:
                img.verify()
            return "OK", None

        elif ext in VIDEO_EXTS:
            cap = cv2.VideoCapture(str(filepath))
            if not cap.isOpened():
                cap.release()
                return "CORRUPT", "cv2 could not open video"
            ret, frame = cap.read()
            cap.release()
            if not ret:
                return "CORRUPT", "video opened but no readable frame"
            return "OK", None

        elif ext in TABULAR_EXTS:
            if ext == ".csv":
                pd.read_csv(filepath, nrows=5)
            else:
                pd.read_excel(filepath, nrows=5)
            return "OK", None

        elif ext in TEXT_LABEL_EXTS:
            if filepath.stat().st_size == 0:
                return "CORRUPT", "empty file"
            with open(filepath, "r", encoding="utf-8", errors="strict") as f:
                f.read()
            return "OK", None

        elif ext in METADATA_EXTS:
            with open(filepath, "r", encoding="utf-8", errors="strict") as f:
                f.read()
            return "METADATA", None

        else:
            return "UNRECOGNIZED", f"unhandled extension: {ext or '(none)'}"

    except Exception as e:
        return "CORRUPT", str(e)


# --- Run integrity check per dataset ---
file_integrity_results = []   # per-dataset summary
file_level_issues = []        # every individual bad file, for the log/report

for row in folder_check_df.itertuples():
    if row.verdict != "OK":
        continue  # skip SKIPPED / MISSING / EMPTY, nothing to check

    dataset_name = row.dataset
    dataset_path = Path(row.path)

    ok_count = 0
    corrupt_count = 0
    metadata_count = 0
    unrecognized_count = 0

    all_files = [f for f in dataset_path.rglob("*") if f.is_file()]

    for f in tqdm(all_files, desc=dataset_name, leave=False):
        status, reason = check_file(f)

        if status == "OK":
            ok_count += 1
        elif status == "METADATA":
            metadata_count += 1
        elif status == "CORRUPT":
            corrupt_count += 1
            logger.warning(f"[CORRUPT] {dataset_name} — {f} — {reason}")
            file_level_issues.append({
                "dataset": dataset_name,
                "file": str(f),
                "issue": reason
            })
        elif status == "UNRECOGNIZED":
            unrecognized_count += 1
            logger.info(f"[UNRECOGNIZED] {dataset_name} — {f} — {reason}")

    logger.info(
        f"[INTEGRITY] {dataset_name} — OK: {ok_count}, CORRUPT: {corrupt_count}, "
        f"METADATA: {metadata_count}, UNRECOGNIZED: {unrecognized_count}"
    )

    file_integrity_results.append({
        "dataset": dataset_name,
        "total_files": len(all_files),
        "ok": ok_count,
        "corrupt": corrupt_count,
        "metadata": metadata_count,
        "unrecognized": unrecognized_count
    })

file_integrity_df = pd.DataFrame(file_integrity_results)
print(file_integrity_df.to_string(index=False))

---

Separate and copy corrupted files into invalid

---

In [ ]:
issues_df = pd.DataFrame(file_level_issues)
copied_invalid = []

for row in issues_df.itertuples():
    dataset_name = row.dataset
    src_path = Path(row.file)

    # Find this dataset's raw root so we can preserve its internal folder structure
    dataset_row = folder_check_df[folder_check_df["dataset"] == dataset_name].iloc[0]
    dataset_raw_root = Path(dataset_row["path"])

    # Path of the file relative to its dataset root (keeps images/labels/... structure intact)
    relative_path = src_path.relative_to(dataset_raw_root)
    dest_path = INVALID_DIR / dataset_name / relative_path

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        shutil.copy2(src_path, dest_path)
        logger.warning(f"[MOVED TO INVALID] {dataset_name} — {relative_path} — reason: {row.issue}")
        copied_invalid.append({
            "dataset": dataset_name,
            "file": str(relative_path),
            "reason": row.issue,
            "dest": str(dest_path)
        })
    except Exception as e:
        logger.error(f"[COPY FAILED] {dataset_name} — {relative_path} — {e}")

copied_invalid_df = pd.DataFrame(copied_invalid)
print(f"Copied {len(copied_invalid_df)} corrupt files into invalid/")
print(copied_invalid_df.groupby("dataset").size().to_string())

In [ ]:
##### structure_survey = []

for row in folder_check_df.itertuples():
    if row.verdict != "OK":
        continue

    dataset_name = row.dataset
    dataset_path = Path(row.path)

    # Top-level subfolders inside this dataset (e.g. images/, labels/, train/, etc.)
    subfolders = sorted([p.name for p in dataset_path.iterdir() if p.is_dir()])

    # Count files by extension at every depth, so we can see if labels exist anywhere at all
    ext_counts = Counter(f.suffix.lower() for f in dataset_path.rglob("*") if f.is_file())
    has_labels = any(ext in {".txt", ".xml", ".json"} for ext in ext_counts)

    structure_survey.append({
        "dataset": dataset_name,
        "subfolders": subfolders if subfolders else "(flat, no subfolders)",
        "has_label_files": has_labels,
        "image_count": ext_counts.get(".jpg", 0) + ext_counts.get(".jpeg", 0) + ext_counts.get(".png", 0),
        "label_count": ext_counts.get(".txt", 0) + ext_counts.get(".xml", 0) + ext_counts.get(".json", 0)
    })

structure_df = pd.DataFrame(structure_survey)
pd.set_option("display.max_colwidth", None)
print(structure_df.to_string(index=False))

---

We then do a pairing check

---

In [ ]:
METADATA_LABEL_FILENAMES = {
    "readme.roboflow.txt", "readme.dataset.txt", "readme.txt",
    "citations.txt", "license.txt", "changelog.txt", "notice.txt"
}

def get_stem_set(folder: Path, exts, exclude_filenames=None):
    exclude_filenames = exclude_filenames or set()
    result = {}
    for f in folder.rglob("*"):
        if f.is_file() and f.suffix.lower() in exts and f.name.lower() not in exclude_filenames:
            result[f.stem] = f
    return result


pairing_results = []
orphan_records = []

corrupt_stems_by_dataset = defaultdict(set)
for issue in file_level_issues:
    p = Path(issue["file"])
    corrupt_stems_by_dataset[issue["dataset"]].add(p.stem)

for row in structure_df.drop_duplicates(subset="dataset").itertuples():
    dataset_name = row.dataset

    if not row.has_label_files:
        pairing_results.append({
            "dataset": dataset_name, "matched": None, "orphan_images": None,
            "orphan_labels": None, "verdict": "NOT_APPLICABLE"
        })
        continue

    if dataset_name == "mapillary_vistas":
        pairing_results.append({
            "dataset": dataset_name, "matched": None, "orphan_images": None,
            "orphan_labels": None, "verdict": "DEFERRED_CUSTOM_CHECK"
        })
        continue

    dataset_path = Path(folder_check_df[folder_check_df["dataset"] == dataset_name]["path"].values[0])

    image_stems = get_stem_set(dataset_path, IMAGE_EXTS)
    label_stems = get_stem_set(dataset_path, {".txt", ".xml"}, exclude_filenames=METADATA_LABEL_FILENAMES)

    corrupt_stems = corrupt_stems_by_dataset.get(dataset_name, set())

    matched = set(image_stems) & set(label_stems)
    matched_clean = matched - corrupt_stems

    orphan_images = set(image_stems) - set(label_stems)
    orphan_labels = set(label_stems) - set(image_stems)

    for stem in orphan_images:
        orphan_records.append({"dataset": dataset_name, "type": "orphan_image", "file": str(image_stems[stem])})
    for stem in orphan_labels:
        orphan_records.append({"dataset": dataset_name, "type": "orphan_label", "file": str(label_stems[stem])})

    logger.info(
        f"[PAIRING] {dataset_name} — matched: {len(matched_clean)}, "
        f"orphan_images: {len(orphan_images)}, orphan_labels: {len(orphan_labels)}"
    )

    pairing_results.append({
        "dataset": dataset_name,
        "matched": len(matched_clean),
        "orphan_images": len(orphan_images),
        "orphan_labels": len(orphan_labels),
        "verdict": "CHECKED"
    })

pairing_df = pd.DataFrame(pairing_results)
print(pairing_df.to_string(index=False))

---

Remove images without label pairs

---

In [ ]:
orphan_df = pd.DataFrame(orphan_records)
copied_orphans = []

for row in orphan_df.itertuples():
    dataset_name = row.dataset
    src_path = Path(row.file)

    dataset_row = folder_check_df[folder_check_df["dataset"] == dataset_name].iloc[0]
    dataset_raw_root = Path(dataset_row["path"])

    relative_path = src_path.relative_to(dataset_raw_root)
    dest_path = INVALID_DIR / dataset_name / relative_path

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        shutil.copy2(src_path, dest_path)
        logger.warning(f"[MOVED TO INVALID] {dataset_name} — {relative_path} — reason: {row.type}")
        copied_orphans.append({
            "dataset": dataset_name,
            "file": str(relative_path),
            "reason": row.type,
            "dest": str(dest_path)
        })
    except Exception as e:
        logger.error(f"[COPY FAILED] {dataset_name} — {relative_path} — {e}")

copied_orphans_df = pd.DataFrame(copied_orphans)
print(f"Copied {len(copied_orphans_df)} orphan files into invalid/")
print(copied_orphans_df.groupby(["dataset", "reason"]).size().to_string())

---

This is a custom check for the mapillary dataset

---

In [ ]:
mapillary_path = Path(folder_check_df[folder_check_df["dataset"] == "mapillary_vistas"]["path"].values[0])

for split in ["training", "validation", "testing"]:
    split_path = mapillary_path / split
    if split_path.exists():
        print(f"\n{split}/ contents:")
        for item in sorted(split_path.iterdir()):
            if item.is_dir():
                file_count = sum(1 for f in item.rglob("*") if f.is_file())
                print(f"  [dir]  {item.name}  ({file_count} files)")
            else:
                print(f"  [file] {item.name}")
    else:
        print(f"\n{split}/ does not exist")

# Also list the top-level json files and any config files
print("\nTop-level files in mapillary_vistas/:")
for item in sorted(mapillary_path.iterdir()):
    if item.is_file():
        print(f"  {item.name}")

---

Check only training and validation folders, skip testing since videos

---

In [ ]:
mapillary_orphans = []
mapillary_summary = []

def stems_in(folder: Path, ext):
    if not folder.exists():
        return set()
    return {f.stem for f in folder.iterdir() if f.is_file() and f.suffix.lower() == ext}

for split in ["training", "validation"]:
    split_path = mapillary_path / split
    image_stems = stems_in(split_path / "images", ".jpg")

    for version in ["v1.2", "v2.0"]:
        version_path = split_path / version
        if not version_path.exists():
            continue

        for subfolder, ext in [("labels", ".png"), ("instances", ".png"), ("panoptic", ".png")]:
            sub_stems = stems_in(version_path / subfolder, ext)
            missing_in_sub = image_stems - sub_stems
            extra_in_sub = sub_stems - image_stems

            for stem in missing_in_sub:
                mapillary_orphans.append({
                    "split": split, "version": version, "subfolder": subfolder,
                    "type": "image_missing_mask", "stem": stem
                })
            for stem in extra_in_sub:
                mapillary_orphans.append({
                    "split": split, "version": version, "subfolder": subfolder,
                    "type": "mask_missing_image", "stem": stem
                })

            mapillary_summary.append({
                "split": split, "version": version, "subfolder": subfolder,
                "images": len(image_stems), "masks": len(sub_stems),
                "missing_masks": len(missing_in_sub), "orphan_masks": len(extra_in_sub)
            })

        # v2.0 also has polygons/ (json, not png)
        if version == "v2.0":
            poly_stems = stems_in(version_path / "polygons", ".json")
            missing_poly = image_stems - poly_stems
            extra_poly = poly_stems - image_stems

            for stem in missing_poly:
                mapillary_orphans.append({
                    "split": split, "version": version, "subfolder": "polygons",
                    "type": "image_missing_mask", "stem": stem
                })
            for stem in extra_poly:
                mapillary_orphans.append({
                    "split": split, "version": version, "subfolder": "polygons",
                    "type": "mask_missing_image", "stem": stem
                })

            mapillary_summary.append({
                "split": split, "version": version, "subfolder": "polygons",
                "images": len(image_stems), "masks": len(poly_stems),
                "missing_masks": len(missing_poly), "orphan_masks": len(extra_poly)
            })

mapillary_summary_df = pd.DataFrame(mapillary_summary)
print(mapillary_summary_df.to_string(index=False))

logger.info(f"[MAPILLARY PAIRING] Checked training/validation across v1.2 and v2.0. "
            f"Total orphan records: {len(mapillary_orphans)}")

---

Annotation check across all dataset

---

In [ ]:
roboflow_datasets = [
    "obstacle_detection_roboflow", "indoor_objects_roboflow", "indoor_detection_vineeth",
    "indoor_objects_5iwhq", "revised_pedestrian_obstacle_detection", "pedestrian_walk",
    "outdoor_objects", "footpath_detection", "stairs_detection"
]

dataset_class_info = {}  # {dataset_name: {"num_classes": int, "names": [...]}}

for dataset_name in roboflow_datasets:
    dataset_path = Path(folder_check_df[folder_check_df["dataset"] == dataset_name]["path"].values[0])
    yaml_candidates = list(dataset_path.glob("*.yaml")) + list(dataset_path.glob("**/data.yaml"))

    if not yaml_candidates:
        logger.warning(f"[CLASS INFO] {dataset_name} — no data.yaml found.")
        continue

    yaml_path = yaml_candidates[0]
    try:
        with open(yaml_path, "r", encoding="utf-8") as f:
            data_yaml = yaml.safe_load(f)

        names = data_yaml.get("names")
        num_classes = data_yaml.get("nc", len(names) if names else None)

        dataset_class_info[dataset_name] = {"num_classes": num_classes, "names": names}
        logger.info(f"[CLASS INFO] {dataset_name} — {num_classes} classes: {names}")

    except Exception as e:
        logger.error(f"[CLASS INFO FAILED] {dataset_name} — {yaml_path} — {e}")

for name, info in dataset_class_info.items():
    print(f"{name}: {info['num_classes']} classes — {info['names']}")

In [ ]:
# --- Fixed YOLO checker (bbox + polygon support) ---
def check_yolo_txt(filepath: Path, num_classes=None):
    issues = []
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
    except Exception as e:
        return [f"unreadable: {e}"]

    if not lines:
        return []

    for i, line in enumerate(lines):
        parts = line.split()
        n = len(parts)

        try:
            class_id = int(float(parts[0]))
        except (ValueError, IndexError):
            issues.append(f"line {i}: non-numeric or missing class_id")
            continue

        if class_id < 0:
            issues.append(f"line {i}: negative class_id {class_id}")
        if num_classes is not None and class_id >= num_classes:
            issues.append(f"line {i}: class_id {class_id} out of range (max {num_classes - 1})")

        coord_values = parts[1:]

        if n == 5:
            try:
                x, y, w, h = (float(v) for v in coord_values)
            except ValueError:
                issues.append(f"line {i}: non-numeric bbox value")
                continue
            if not (0 <= x <= 1 and 0 <= y <= 1):
                issues.append(f"line {i}: center out of bounds ({x}, {y})")
            if not (0 < w <= 1 and 0 < h <= 1):
                issues.append(f"line {i}: invalid width/height ({w}, {h})")

        elif n >= 7 and n % 2 == 1:
            try:
                coords = [float(v) for v in coord_values]
            except ValueError:
                issues.append(f"line {i}: non-numeric polygon coordinate")
                continue
            out_of_bounds = [c for c in coords if not (0 <= c <= 1)]
            if out_of_bounds:
                issues.append(f"line {i}: {len(out_of_bounds)} polygon coordinate(s) out of 0-1 bounds")

        else:
            issues.append(f"line {i}: unexpected field count {n} (not bbox-5 or valid polygon)")

    return issues


# --- Re-run annotation check with console logging silenced ---
annotation_issues = []
annotation_summary = []

stream_handler.setLevel(logging.CRITICAL)  # <-- silence console, file logging still works

try:
    for row in structure_df.drop_duplicates(subset="dataset").itertuples():
        dataset_name = row.dataset

        if not row.has_label_files or dataset_name == "mapillary_vistas":
            continue

        dataset_path = Path(folder_check_df[folder_check_df["dataset"] == dataset_name]["path"].values[0])
        class_info = dataset_class_info.get(dataset_name)
        num_classes = class_info["num_classes"] if class_info else None

        label_files = [f for f in dataset_path.rglob("*")
                       if f.is_file() and f.suffix.lower() in {".txt", ".xml"}
                       and f.name.lower() not in METADATA_LABEL_FILENAMES]

        clean_count = 0
        bad_count = 0

        for f in tqdm(label_files, desc=dataset_name, leave=False):
            if f.suffix.lower() == ".txt":
                issues = check_yolo_txt(f, num_classes=num_classes)
            else:
                issues = check_voc_xml(f)

            if issues:
                bad_count += 1
                for issue in issues:
                    annotation_issues.append({"dataset": dataset_name, "file": str(f), "issue": issue})
                logger.warning(f"[BAD ANNOTATION] {dataset_name} — {f.name} — {issues[0]}")
            else:
                clean_count += 1

        logger.info(f"[ANNOTATION CHECK] {dataset_name} — clean: {clean_count}, bad: {bad_count}")

        annotation_summary.append({
            "dataset": dataset_name,
            "total_labels": len(label_files),
            "clean": clean_count,
            "bad": bad_count,
            "class_range_checked": num_classes is not None
        })

finally:
    stream_handler.setLevel(logging.INFO)  # <-- always restore, even if something errors

annotation_summary_df = pd.DataFrame(annotation_summary)
print(annotation_summary_df.to_string(index=False))

---

Inspect found issues

---

In [ ]:
issues_df = pd.DataFrame(annotation_issues)
print(issues_df.to_string(index=False))

---

Remove those issues along with their pictures (only use when the number is not too big)

---

In [ ]:
# --- Step: Copy files with bad annotation content into invalid/ (whole file + paired image) ---

bad_annotation_df = pd.DataFrame(annotation_issues)
# One row per unique bad label file (a file might have multiple bad lines, only need to handle it once)
bad_label_files = bad_annotation_df.drop_duplicates(subset=["dataset", "file"])

copied_bad_annotations = []

for row in bad_label_files.itertuples():
    dataset_name = row.dataset
    label_path = Path(row.file)

    dataset_row = folder_check_df[folder_check_df["dataset"] == dataset_name].iloc[0]
    dataset_raw_root = Path(dataset_row["path"])

    # Copy the bad label file itself
    relative_label_path = label_path.relative_to(dataset_raw_root)
    dest_label_path = INVALID_DIR / dataset_name / relative_label_path
    dest_label_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        shutil.copy2(label_path, dest_label_path)
        logger.warning(f"[MOVED TO INVALID] {dataset_name} — {relative_label_path} — reason: bad annotation content")
        copied_bad_annotations.append({
            "dataset": dataset_name, "file": str(relative_label_path),
            "type": "label", "reason": "bad annotation content"
        })
    except Exception as e:
        logger.error(f"[COPY FAILED] {dataset_name} — {relative_label_path} — {e}")

    # Find and copy the paired image (same stem, look in IMAGE_EXTS within the same dataset)
    stem = label_path.stem
    matching_images = [f for f in dataset_raw_root.rglob(f"{stem}.*") if f.suffix.lower() in IMAGE_EXTS]

    for img_path in matching_images:
        relative_img_path = img_path.relative_to(dataset_raw_root)
        dest_img_path = INVALID_DIR / dataset_name / relative_img_path
        dest_img_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            shutil.copy2(img_path, dest_img_path)
            logger.warning(f"[MOVED TO INVALID] {dataset_name} — {relative_img_path} — reason: paired label has bad annotation content")
            copied_bad_annotations.append({
                "dataset": dataset_name, "file": str(relative_img_path),
                "type": "image", "reason": "paired label has bad annotation content"
            })
        except Exception as e:
            logger.error(f"[COPY FAILED] {dataset_name} — {relative_img_path} — {e}")

copied_bad_annotations_df = pd.DataFrame(copied_bad_annotations)
print(f"Copied {len(copied_bad_annotations_df)} files (labels + paired images) into invalid/")
print(copied_bad_annotations_df.groupby(["dataset", "type"]).size().to_string())

---

Inspect the mapillary structure again

---

In [ ]:
import json

# Inspect one panoptic JSON file structure
panoptic_json_path = mapillary_path / "training" / "v2.0" / "panoptic" / "panoptic_2020.json"

with open(panoptic_json_path, "r", encoding="utf-8") as f:
    panoptic_data = json.load(f)

print("Top-level keys:", list(panoptic_data.keys()))
print("\nNumber of annotations:", len(panoptic_data.get("annotations", [])))
print("Number of categories:", len(panoptic_data.get("categories", [])))
print("\nSample annotation entry:")
print(json.dumps(panoptic_data["annotations"][0], indent=2)[:1000])
print("\nSample category entry:")
print(json.dumps(panoptic_data["categories"][0], indent=2))

---

The mapillary annotation check (I hate whoever gave me this)

---

In [ ]:
# --- Step: Validate mapillary_vistas panoptic JSON content ---

mapillary_json_issues = []
mapillary_json_summary = []

def check_panoptic_json(json_path: Path, split_path: Path, version: str, split: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images = data.get("images", [])
    annotations = data.get("annotations", [])
    categories = data.get("categories", [])

    valid_category_ids = {c["id"] for c in categories}
    image_ids_in_json = {img["id"] for img in images}
    image_filenames_in_json = {img["file_name"] for img in images}

    # What actually exists on disk
    actual_image_files = {f.name for f in (split_path / "images").glob("*.jpg")}

    issues = []

    # 1. Every image entry's file_name should exist on disk
    missing_on_disk = image_filenames_in_json - actual_image_files
    for fname in missing_on_disk:
        issues.append({"type": "image_listed_but_missing_on_disk", "file": fname})

    # 2. Every actual image file should be listed in the JSON
    unlisted_files = actual_image_files - image_filenames_in_json
    for fname in unlisted_files:
        issues.append({"type": "image_on_disk_but_not_in_json", "file": fname})

    # 3. Every annotation's image_id should correspond to a real image entry
    annotation_image_ids = {a["image_id"] for a in annotations}
    orphan_annotations = annotation_image_ids - image_ids_in_json
    for img_id in orphan_annotations:
        issues.append({"type": "annotation_references_unknown_image_id", "file": img_id})

    # 4. Every image should have exactly one annotation entry
    images_without_annotation = image_ids_in_json - annotation_image_ids
    for img_id in images_without_annotation:
        issues.append({"type": "image_has_no_annotation_entry", "file": img_id})

    # 5. Every category_id used inside segments_info should exist in categories
    bad_category_refs = 0
    bad_bbox_count = 0
    for ann in annotations:
        for seg in ann.get("segments_info", []):
            if seg["category_id"] not in valid_category_ids:
                bad_category_refs += 1
                issues.append({
                    "type": "invalid_category_id",
                    "file": f"{ann['image_id']} segment {seg['id']} category {seg['category_id']}"
                })
            bbox = seg.get("bbox", [])
            if len(bbox) == 4:
                x, y, w, h = bbox
                if w <= 0 or h <= 0 or x < 0 or y < 0:
                    bad_bbox_count += 1
                    issues.append({
                        "type": "invalid_segment_bbox",
                        "file": f"{ann['image_id']} segment {seg['id']}"
                    })

    for issue in issues:
        mapillary_json_issues.append({
            "split": split, "version": version, **issue
        })

    mapillary_json_summary.append({
        "split": split, "version": version,
        "total_images": len(images), "total_annotations": len(annotations),
        "missing_on_disk": len(missing_on_disk),
        "unlisted_files": len(unlisted_files),
        "orphan_annotations": len(orphan_annotations),
        "images_without_annotation": len(images_without_annotation),
        "invalid_category_refs": bad_category_refs,
        "invalid_bboxes": bad_bbox_count
    })


json_files_to_check = [
    ("training", "v1.2", mapillary_path / "training" / "v1.2" / "panoptic" / "panoptic_2018.json"),
    ("training", "v2.0", mapillary_path / "training" / "v2.0" / "panoptic" / "panoptic_2020.json"),
    ("validation", "v1.2", mapillary_path / "validation" / "v1.2" / "panoptic" / "panoptic_2018.json"),
    ("validation", "v2.0", mapillary_path / "validation" / "v2.0" / "panoptic" / "panoptic_2020.json"),
]

stream_handler.setLevel(logging.CRITICAL)
try:
    for split, version, json_path in json_files_to_check:
        if json_path.exists():
            check_panoptic_json(json_path, mapillary_path / split, version, split)
        else:
            logger.warning(f"[MAPILLARY JSON] Missing expected file: {json_path}")
finally:
    stream_handler.setLevel(logging.INFO)

mapillary_json_summary_df = pd.DataFrame(mapillary_json_summary)
print(mapillary_json_summary_df.to_string(index=False))

---

Inspect bboxes issue

---

In [ ]:
# --- Inspect actual invalid bbox entries ---
bbox_issues_df = pd.DataFrame(mapillary_json_issues)
bbox_only = bbox_issues_df[bbox_issues_df["type"] == "invalid_segment_bbox"]

print("Sample invalid bbox issues:")
print(bbox_only.head(10).to_string(index=False))

# Let's pull up the actual segment data for a couple of these to see the real bbox values
with open(mapillary_path / "training" / "v2.0" / "panoptic" / "panoptic_2020.json", "r", encoding="utf-8") as f:
    v2_data = json.load(f)

# Grab the first flagged image_id and inspect its segments
first_bad = bbox_only.iloc[0]["file"].split(" segment ")[0]
matching_ann = next(a for a in v2_data["annotations"] if a["image_id"] == first_bad)

print(f"\nFull segments_info for image_id {first_bad}:")
for seg in matching_ann["segments_info"]:
    print(seg)

---

These zero-area bboxes come from tiny (often 1-pixel) panoptic segments, not corrupted data — this code logs them for the record but doesn't treat them as invalid, so the images and their annotations still go into validated

---

In [ ]:
# --- Reclassify tiny/zero-area bboxes as informational, not invalid ---

bbox_issues = [i for i in mapillary_json_issues if i["type"] == "invalid_segment_bbox"]

logger.info(
    f"[MAPILLARY BBOX] {len(bbox_issues)} segments flagged with zero/near-zero area bbox "
    f"across v2.0 (training + validation). These are sub-pixel panoptic boundary artifacts, "
    f"not treated as validation failures — images and their annotations remain in the "
    f"validated pipeline. See validation_log for full list if needed."
)

print(f"{len(bbox_issues)} zero-area bbox segments noted as informational (not invalid).")
print("These images and annotations will proceed to validated/ normally.")

---

We now check for duplication, only within each dataset, cross checking on large scale will be done when merging

---

In [ ]:
# --- Step: Exact duplicate detection, within each dataset ---

def hash_file(filepath: Path, chunk_size=8192):
    """Compute MD5 hash of a file's contents."""
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


duplicate_results = []
duplicate_records = []  # every individual duplicate file, for the report

stream_handler.setLevel(logging.CRITICAL)
try:
    for row in folder_check_df.itertuples():
        if row.verdict != "OK":
            continue

        dataset_name = row.dataset
        dataset_path = Path(row.path)

        # Only hash actual data files — skip metadata (README, yaml, etc.)
        all_files = [f for f in dataset_path.rglob("*")
                     if f.is_file() and f.name.lower() not in METADATA_LABEL_FILENAMES
                     and f.suffix.lower() not in METADATA_EXTS]

        hash_to_files = defaultdict(list)

        for f in tqdm(all_files, desc=dataset_name, leave=False):
            try:
                file_hash = hash_file(f)
                hash_to_files[file_hash].append(f)
            except Exception as e:
                logger.error(f"[HASH FAILED] {dataset_name} — {f} — {e}")

        # Any hash with more than 1 file = duplicates
        dup_groups = {h: files for h, files in hash_to_files.items() if len(files) > 1}
        total_dup_files = sum(len(files) - 1 for files in dup_groups.values())  # extras, keeping 1 as "original"

        for file_hash, files in dup_groups.items():
            # Keep the first one (by path sort) as "original", rest are duplicates
            sorted_files = sorted(files)
            original = sorted_files[0]
            for dup in sorted_files[1:]:
                duplicate_records.append({
                    "dataset": dataset_name,
                    "original": str(original),
                    "duplicate": str(dup),
                    "hash": file_hash
                })
                logger.warning(f"[DUPLICATE] {dataset_name} — {dup} — duplicate of {original}")

        logger.info(f"[DUPLICATE CHECK] {dataset_name} — total files: {len(all_files)}, "
                    f"duplicate groups: {len(dup_groups)}, extra duplicate files: {total_dup_files}")

        duplicate_results.append({
            "dataset": dataset_name,
            "total_files_checked": len(all_files),
            "duplicate_groups": len(dup_groups),
            "extra_duplicate_files": total_dup_files
        })
finally:
    stream_handler.setLevel(logging.INFO)

duplicate_summary_df = pd.DataFrame(duplicate_results)
print(duplicate_summary_df.to_string(index=False))

---

Mark files for removal

---

In [ ]:
# --- Step: Resolve cross-split duplicates in obstacle_detection_kaggle ---
# Rule: keep the copy in 'train' if one exists; otherwise keep 'test' over 'valid' (arbitrary tiebreak)

split_priority = {"train": 0, "test": 1, "valid": 2}  # lower number = higher priority to KEEP

cross_split_dups = kaggle_dups[kaggle_dups["cross_split"]].copy()

removed_for_leakage = []

for row in cross_split_dups.itertuples():
    orig_path = Path(row.original)
    dup_path = Path(row.duplicate)

    orig_priority = split_priority.get(row.original_split, 99)
    dup_priority = split_priority.get(row.duplicate_split, 99)

    # The one with LOWER priority number gets kept; the other gets removed (copied to invalid, not deleted from raw)
    if orig_priority <= dup_priority:
        keep, remove = orig_path, dup_path
    else:
        keep, remove = dup_path, orig_path

    removed_for_leakage.append({
        "dataset": "obstacle_detection_kaggle",
        "kept": str(keep),
        "removed": str(remove),
        "reason": "cross-split duplicate (data leakage risk)"
    })

removed_df = pd.DataFrame(removed_for_leakage).drop_duplicates(subset="removed")
print(f"Files to remove from validated pool due to cross-split leakage: {len(removed_df)}")
print(removed_df["removed"].apply(lambda p: get_split(p)).value_counts())

---

Copy those into invalid

---

In [ ]:
# --- Step: Copy cross-split duplicate images + paired labels into invalid/ ---

dataset_raw_root = Path(folder_check_df[folder_check_df["dataset"] == "obstacle_detection_kaggle"]["path"].values[0])

copied_leakage = []

for row in removed_df.itertuples():
    img_path = Path(row.removed)

    # Copy the duplicate image itself
    relative_img_path = img_path.relative_to(dataset_raw_root)
    dest_img_path = INVALID_DIR / "obstacle_detection_kaggle" / relative_img_path
    dest_img_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        shutil.copy2(img_path, dest_img_path)
        logger.warning(f"[MOVED TO INVALID] obstacle_detection_kaggle — {relative_img_path} — reason: cross-split duplicate")
        copied_leakage.append({"file": str(relative_img_path), "type": "image"})
    except Exception as e:
        logger.error(f"[COPY FAILED] obstacle_detection_kaggle — {relative_img_path} — {e}")

    # Find and copy the paired label (same stem, .txt, in the corresponding labels/ folder)
    # Structure: .../images/IMG_X.jpg -> .../labels/IMG_X.txt
    label_path = Path(str(img_path).replace("\\images\\", "\\labels\\")).with_suffix(".txt")

    if label_path.exists():
        relative_label_path = label_path.relative_to(dataset_raw_root)
        dest_label_path = INVALID_DIR / "obstacle_detection_kaggle" / relative_label_path
        dest_label_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            shutil.copy2(label_path, dest_label_path)
            logger.warning(f"[MOVED TO INVALID] obstacle_detection_kaggle — {relative_label_path} — reason: paired image is cross-split duplicate")
            copied_leakage.append({"file": str(relative_label_path), "type": "label"})
        except Exception as e:
            logger.error(f"[COPY FAILED] obstacle_detection_kaggle — {relative_label_path} — {e}")
    else:
        logger.warning(f"[NO PAIRED LABEL] obstacle_detection_kaggle — {img_path} — expected label not found at {label_path}")

copied_leakage_df = pd.DataFrame(copied_leakage)
print(f"Copied {len(copied_leakage_df)} files (images + labels) into invalid/ due to cross-split leakage")
print(copied_leakage_df["type"].value_counts())

---

Final confirmation against yaml

---

In [ ]:
# --- Step 6: Formal cross-check against download_sheet.yaml ---

yaml_check_results = []

for name, meta in datasets.items():
    status = meta.get("status")
    source = meta.get("source")
    folder_name = meta.get("folder")
    license_info = meta.get("ethics", {}).get("license")

    entry = {
        "dataset": name,
        "yaml_status": status,
        "yaml_source": source,
        "yaml_folder": folder_name,
        "yaml_license": license_info,
    }

    if status == "rejected":
        entry["pipeline_action"] = "SKIPPED"
        entry["notes"] = "Correctly excluded — not downloaded/validated per YAML status."
        yaml_check_results.append(entry)
        continue

    # Confirm this dataset actually went through folder + integrity checks
    in_folder_check = name in folder_check_df["dataset"].values
    in_integrity_check = name in file_integrity_df["dataset"].values if "file_integrity_df" in dir() else False

    folder_row = folder_check_df[folder_check_df["dataset"] == name]
    actual_verdict = folder_row["verdict"].values[0] if len(folder_row) else "NOT_FOUND"
    expected_path_exists = actual_verdict == "OK"

    entry["pipeline_action"] = "VALIDATED" if expected_path_exists else "MISSING/INCOMPLETE"
    entry["in_folder_check"] = in_folder_check
    entry["in_integrity_check"] = in_integrity_check
    entry["folder_verdict"] = actual_verdict

    if not expected_path_exists:
        entry["notes"] = "MISMATCH — approved in YAML but not found/processed correctly."
        logger.error(f"[YAML CROSS-CHECK] {name} — approved in YAML but folder verdict is '{actual_verdict}'")
    else:
        entry["notes"] = "OK — matches YAML, processed through pipeline."

    yaml_check_results.append(entry)

yaml_check_df = pd.DataFrame(yaml_check_results)
print(yaml_check_df.to_string(index=False))

# Explicit final tally
approved_count = sum(1 for m in datasets.values() if m.get("status") == "approved")
rejected_count = sum(1 for m in datasets.values() if m.get("status") == "rejected")
processed_ok = (yaml_check_df["pipeline_action"] == "VALIDATED").sum()

print(f"\nYAML total datasets   : {len(datasets)}")
print(f"Approved in YAML      : {approved_count}")
print(f"Rejected/skipped      : {rejected_count}")
print(f"Successfully processed: {processed_ok}")
print(f"Mismatch/missing      : {approved_count - processed_ok}")

logger.info(f"[YAML CROSS-CHECK COMPLETE] {processed_ok}/{approved_count} approved datasets fully processed. "
            f"{rejected_count} correctly skipped.")

---

Empty the validated folder before moving

---

In [ ]:
if VALIDATED_DIR.exists():
    shutil.rmtree(VALIDATED_DIR)
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

print("validated/ cleared and recreated.")

---

Copy non-invalid data into validated folder

---

In [ ]:
validated_copy_log = []

stream_handler.setLevel(logging.CRITICAL)
try:
    for row in folder_check_df.itertuples():
        if row.verdict != "OK":
            continue

        dataset_name = row.dataset
        dataset_path = Path(row.path)
        source = datasets[dataset_name]["source"]

        all_files = [f for f in dataset_path.rglob("*") if f.is_file()]

        copied = 0
        skipped_metadata = 0
        skipped_invalid = 0

        for f in tqdm(all_files, desc=dataset_name, leave=False):
            if is_metadata(f):
                skipped_metadata += 1
                continue
            if str(f.resolve()) in excluded_paths:
                skipped_invalid += 1
                continue

            relative_path = f.relative_to(dataset_path)
            dest_path = VALIDATED_DIR / source / dataset_name / relative_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)

            try:
                shutil.copy2(f, dest_path)
                copied += 1
            except Exception as e:
                logger.error(f"[VALIDATED COPY FAILED] {dataset_name} — {f} — {e}")

        logger.info(f"[VALIDATED] {dataset_name} — copied: {copied}, "
                    f"skipped_metadata: {skipped_metadata}, skipped_invalid: {skipped_invalid}")

        validated_copy_log.append({
            "dataset": dataset_name,
            "total_files": len(all_files),
            "copied_to_validated": copied,
            "skipped_metadata": skipped_metadata,
            "skipped_invalid": skipped_invalid
        })
finally:
    stream_handler.setLevel(logging.INFO)

validated_copy_df = pd.DataFrame(validated_copy_log)
print(validated_copy_df.to_string(index=False))

---

Verification of the file moving (you might notice some missing files, yes, those are README and some unnecessary stuff)

---

In [ ]:
# --- Verify actual file counts on disk in validated/ and invalid/ ---

validated_actual = sum(1 for f in VALIDATED_DIR.rglob("*") if f.is_file())
invalid_actual = sum(1 for f in INVALID_DIR.rglob("*") if f.is_file())
raw_total = sum(1 for f in RAW_DIR.rglob("*") if f.is_file())

print(f"raw/       : {raw_total} files")
print(f"validated/ : {validated_actual} files")
print(f"invalid/   : {invalid_actual} files")
print(f"validated + invalid = {validated_actual + invalid_actual}")
print(f"(raw includes huggingface's empty folder + all metadata files not copied anywhere)")

---

Report consolidation and log closing

---

In [ ]:
report_rows = []

for row in folder_check_df.itertuples():
    dataset_name = row.dataset

    if row.verdict == "SKIPPED":
        report_rows.append({
            "dataset": dataset_name,
            "yaml_status": "rejected",
            "source": None,
            "license": None,
            "total_raw_files": None,
            "validated_files": None,
            "invalid_files": None,
            "metadata_files": None,
            "verdict": "SKIPPED"
        })
        continue

    meta = datasets[dataset_name]
    vc_row = validated_copy_df[validated_copy_df["dataset"] == dataset_name]

    total_raw = vc_row["total_files"].values[0] if len(vc_row) else None
    copied_validated = vc_row["copied_to_validated"].values[0] if len(vc_row) else None
    skipped_meta = vc_row["skipped_metadata"].values[0] if len(vc_row) else None
    skipped_inv = vc_row["skipped_invalid"].values[0] if len(vc_row) else None

    report_rows.append({
        "dataset": dataset_name,
        "yaml_status": meta.get("status"),
        "source": meta.get("source"),
        "license": meta.get("ethics", {}).get("license"),
        "total_raw_files": total_raw,
        "validated_files": copied_validated,
        "invalid_files": skipped_inv,
        "metadata_files": skipped_meta,
        "verdict": "PROCESSED"
    })

final_report_df = pd.DataFrame(report_rows)

# Write the summary CSV
report_path = REPORT_DIR / f"validation_report_{timestamp}.csv"
final_report_df.to_csv(report_path, index=False)

# Write the detailed issue-level CSV (every individual flagged file, across all checks)
all_issues = []

for issue in file_level_issues:
    all_issues.append({"dataset": issue["dataset"], "file": issue["file"], "category": "corrupt", "reason": issue["issue"]})
for rec in orphan_records:
    all_issues.append({"dataset": rec["dataset"], "file": rec["file"], "category": "orphan", "reason": rec["type"]})
for row in bad_label_files.itertuples():
    all_issues.append({"dataset": row.dataset, "file": row.file, "category": "bad_annotation", "reason": "invalid annotation content"})
for row in removed_df.itertuples():
    all_issues.append({"dataset": "obstacle_detection_kaggle", "file": row.removed, "category": "cross_split_duplicate", "reason": f"duplicate, kept: {row.kept}"})

all_issues_df = pd.DataFrame(all_issues)
issues_path = REPORT_DIR / f"validation_issues_{timestamp}.csv"
all_issues_df.to_csv(issues_path, index=False)

# Final log entries
logger.info("=" * 60)
logger.info("VALIDATION RUN COMPLETE")
logger.info(f"Total datasets in YAML       : {len(datasets)}")
logger.info(f"Approved and processed       : {(final_report_df['verdict'] == 'PROCESSED').sum()}")
logger.info(f"Skipped (rejected in YAML)   : {(final_report_df['verdict'] == 'SKIPPED').sum()}")
logger.info(f"Total files in validated/    : {final_report_df['validated_files'].sum()}")
logger.info(f"Total files in invalid/      : {int(final_report_df['invalid_files'].sum())}")
logger.info(f"Summary report written to    : {report_path}")
logger.info(f"Detailed issues written to   : {issues_path}")
logger.info("=" * 60)

# Close file handler cleanly
file_handler.close()
logger.removeHandler(file_handler)

print("Validation complete.")
print(f"\nSummary report : {report_path}")
print(f"Detailed issues: {issues_path}")
print(f"\n{final_report_df.to_string(index=False)}")

---

And that's the end of the validation process, this line conclude the notebook and one's can move onto the next one.
There might be some code duplication such as the creation of the folders from the collection step, but I'll complete it along the way as soon as the dataset is ready.

---